# 03 · Preprocesado común: oraciones, tokens y raíces

Tercer notebook del preprocesado (E2). Parte de los cuatro ficheros `*_limpio.jsonl` de los notebooks 01 y 02 y les aplica a todos **la misma función de preprocesado**:

1. Normalización Unicode (NFC).
2. División en oraciones.
3. Tokenización (`tokens`, para embeddings).
4. Raíces sin stopwords (`raices_bow`, para BoW / TF-IDF).

El resultado son los `*_procesado.jsonl`, con el formato de `ESQUEMA_DATOS.md`.

## 0. Preparación

In [ ]:
import json
import re
import unicodedata
from pathlib import Path

import pandas as pd
from nltk.tokenize import sent_tokenize

# El notebook se ejecuta desde la carpeta E2/, por eso las rutas empiezan por "..".
CARPETA_DATOS = Path("../data/processed")
FICHEROS = ["fallacyes_proto", "fallacyes_espont", "fallacyes_political", "congreso"]

corpus = {
    nombre: pd.read_json(CARPETA_DATOS / f"{nombre}_limpio.jsonl", lines=True, dtype=False)
    for nombre in FICHEROS
}
for nombre, tabla in corpus.items():
    print(f"{nombre}: {len(tabla)} textos")

## 1. Normalización Unicode (NFC)

Una letra con tilde se puede guardar de dos formas: como un solo carácter (`é`) o como la letra más una tilde suelta (`e` + `´`). Se ven igual, pero para Python son cadenas distintas, y el Word2Vec las trataría como dos palabras. NFC lo deja todo en la primera forma. **No quita las tildes.**

In [ ]:
compuesta = "país"  # "i" + tilde suelta
print(compuesta, "==", "país", "→", compuesta == "país")
print("con NFC →", unicodedata.normalize("NFC", compuesta) == "país")

for nombre, tabla in corpus.items():
    cambian = tabla["texto"].map(lambda texto: unicodedata.normalize("NFC", texto) != texto)
    print(f"{nombre}: textos que cambian con NFC: {cambian.sum()}")

En nuestros datos no cambia ningún texto. Se aplica igualmente: es una red de seguridad para textos nuevos (copiados de un PDF o escritos en un Mac) y es una línea de código.

## 2. División en oraciones: qué falla con `sent_tokenize`

`sent_tokenize` de NLTK usa el modelo Punkt para español. Punkt no corta en todos los puntos: tiene una lista de abreviaturas tras las que no corta. Se mira dónde se equivoca en nuestros datos.

In [ ]:
pruebas = [
    "La Sra. García dijo que no. El Sr. López tampoco.",
    "Lo dijo el Prof. Martínez en clase.",
    "Vive en St. Louis desde hace años.",
    "Lo aprobó EE. UU. y luego Europa.",
    "Según el art. 155 de la Constitución.",
    "Vinieron todos: Juan, Ana, etc. Pero nadie habló.",
]
for prueba in pruebas:
    print(sent_tokenize(prueba, language="spanish"))

In [ ]:
# Cuántas veces sale cada abreviatura en cada corpus
ABREVIATURAS_A_BUSCAR = {
    "Sra.": r"\bSra\.", "Prof.": r"\bProf\.", "Dr. / Dra.": r"\bDra?\.", "St.": r"\bSt\.",
    "art.": r"\barts?\.", "núm.": r"\bnúm\.", "cap.": r"\bcap\.", "vol.": r"\bvol\.",
    "EE. UU.": r"EE\. ?UU\.", "etc.": r"\betc\.",
}
pd.DataFrame({
    nombre: {abreviatura: tabla["texto"].str.count(patron).sum() for abreviatura, patron in ABREVIATURAS_A_BUSCAR.items()}
    for nombre, tabla in corpus.items()
})

### 2.1 Oraciones muy largas

El texto de Political viene de una transcripción automática (WhisperX), que a veces puntúa poco. Se mira cuánto miden las oraciones de cada corpus.

In [ ]:
oraciones_de_prueba = {
    nombre: tabla["texto"].map(lambda texto: sent_tokenize(texto, language="spanish")).explode()
    for nombre, tabla in corpus.items()
}
pd.DataFrame({
    nombre: {
        "oraciones": len(oraciones),
        "palabras (mediana)": oraciones.str.split().str.len().median(),
        "palabras (percentil 95)": oraciones.str.split().str.len().quantile(0.95),
        "palabras (máximo)": oraciones.str.split().str.len().max(),
        "más de 80 palabras": (oraciones.str.split().str.len() > 80).sum(),
    }
    for nombre, oraciones in oraciones_de_prueba.items()
})

In [ ]:
# Ejemplos de oraciones de más de 80 palabras
for nombre in ["fallacyes_political", "congreso"]:
    oraciones = oraciones_de_prueba[nombre]
    largas = oraciones[oraciones.str.split().str.len() > 80]
    print(f"--- {nombre}")
    for oracion in largas.sample(2, random_state=1):
        print("   ", oracion[:350], "...\n")